In [1]:
# Run this cell only once
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path
# samantha_path = Path('.').absolute().parent.parent.parent
samantha_path = Path('/mnt/bn/ashaw-us/repos/comparison/samantha')
print('Samantha path:', samantha_path)
assert samantha_path.name == 'samantha', "Must set semantha_path to project root."
os.chdir(str(samantha_path))

Samantha path: /mnt/bn/ashaw-us/repos/comparison/samantha


In [2]:
from IPython.display import Audio
from recipes.musiclm.inference.utils import save_wav, slugify

# Create dataset

In [3]:
from recipes.musiclm.datasets.inference import InferenceDataset

In [4]:
# # # Load existing prompts
ds = InferenceDataset.from_prompt_path("/mnt/bn/audio-diffusion/data/google_prompts/text_prompt_collection_20230615.csv")
prompt_texts = [item['text'] for item in ds.items]
prompt_texts[:3]

['Acoustic guitar', 'Electric guitar', 'Cello']

In [5]:
# # Run user defined prompts
# prompt_texts = [
#     "Melodic techno", "Acoustic guitar", "Create a hyped Drake beat with heavy 808 bass"
# ]

## Load from RLHF

In [ ]:
from recipes.musiclm.lightning.rlhf import SemanticSequenceTrainingModule

In [6]:
ckpt_path = "/mnt/bn/audio-diffusion/ashaw/logs/test/semantic_flash_llama_rlhf/mcc40m_filtered_vad_seqtrain_diffusion/checkpoints/step=000000.ckpt"

In [ ]:
rlhf_model = SemanticSequenceTrainingModule.load_from_checkpoint(ckpt_path)
rlhf_model.load_required_modules()
rlhf_model.eval()
rlhf_model.cuda()

# Generate

In [ ]:
wavs = rlhf_model.generate_audio(prompt_texts, hp=None).cpu()

# Eval

In [ ]:
Audio(wavs[0], rate=24000)

### Save to zip

In [22]:
output_dir = Path('/tmp/rlhf')

In [ ]:
for i, (wav, prompt_text) in enumerate(zip(wavs, prompt_texts)):
    os.makedirs(output_dir, exist_ok=True)
    fp = os.path.join(output_dir, f"{slugify(prompt_text)[:128]}.{i}.wav")
    print(f"[Saving] {fp}")
    save_wav(wav, fp, sr=24000)


In [31]:
parent_path = str(output_dir.absolute().parent)
dir_name = output_dir.name
!cd "$parent_path" && zip -r "$dir_name".zip "$dir_name"
!echo Path: "$parent_path"/"$dir_name".zip

updating: rlhf/ (stored 0%)
updating: rlhf/create-a-hyped-drake-beat-with-heavy-808-bass.2.wav (deflated 9%)
updating: rlhf/acoustic-guitar.1.wav (deflated 9%)
updating: rlhf/melodic-techno.0.wav (deflated 9%)
updating: rlhf/create-a-hyped-drake-beat-with-heavy-808-bass.<built-in function round>.wav (deflated 9%)
updating: rlhf/acoustic-guitar.<built-in function round>.wav (deflated 9%)
updating: rlhf/melodic-techno.<built-in function round>.wav (deflated 9%)
Path: /tmp/rlhf.zip
